# 大模型推理与 Serving



## 问题路线

1. [用户说“慢”，我们究竟要解释哪一段等待？](#question-01)
2. [为什么输入变长，会拖慢第一个 token？](#question-02)
3. [工作变多，就一定是 GPU 算力不够吗？](#question-03)
4. [不保存完整 Attention 矩阵，结果还能一样吗？](#question-04)
5. [有了 KV Cache，后续生成为什么仍有成本？](#question-05)
6. [显存总量够，为什么还要把 Cache 分页？](#question-06)
7. [相同的开头能复用吗，缓存又该留多久？](#question-07)
8. [把数值压成低比特，为什么不一定更快？](#question-08)
9. [模型给出 logits 后，下一个 token 是怎么选出的？](#question-09)
10. [能否一次验证多个 token，而不改变目标分布？](#question-10)
11. [单请求快了，为什么多人同时用反而更慢？](#question-11)
12. [长输入怎样给正在生成的请求让出执行机会？](#question-12)
13. [拆开 Prefill 与 Decode，省下的时间会被传输吃掉吗？](#question-13)
14. [吞吐提高了，能否直接宣布优化成功？](#question-14)
15. [怎样做实验，才能知道究竟是哪项改动有效？](#question-15)
16. [面对新的慢请求，第一项证据该查什么？](#question-16)

[运行边界自检](#selfcheck)



```text
请求进入 → 排队/输入准备 → Prefill → 首 token → Decode 循环 → 输出完成
                            │                    ↑   │
                            └── 写入 KV Cache ───┘   └── 读取并追加 KV
```

图中每一段都可能影响等待，但负责它们的对象不同：模型决定计算图、头数与状态表示；runtime/backend 决定 loader、kernel、Cache 实现与执行路径；scheduler 决定队列、批次、接纳和资源池；硬件提供计算、显存、带宽和互联。后文每次选择优化，都要先确认它改变的是哪一个对象。


<a id="question-01"></a>

## 01｜用户说“慢”，我们究竟要解释哪一段等待？

用户提交一段输入，等到第一个字出现，再看回答逐步生成，最后收到完成通知。这里至少有两种体验：迟迟不开口，以及开口之后断断续续。两者不能用一个总耗时解释。

输入 4096 token，生成 128 token。假设首 token 等了 200 ms，随后平均每 token 用 20 ms，且末 token 与完成事件同时到达，那么总等待是 `200 + 127×20 = 2740 ms`。把首 token 等待减半只省 100 ms；把后续间隔减半却省 1270 ms。若用户只关心何时开始响应，判断又会不同。因此第一步是定服务目标，再拆时间。


Prefill 批量处理已知输入，并产生后续生成需要的 KV。通常首 token 可由最后一个输入位置的 logits 选出；后续 Decode 把新 token 输入模型，读取历史 KV、计算 logits、选择输出，并继续追加状态。KV Cache 是贯穿阶段的状态对象，不应在 Prefill 和 Decode 中间画成一个独立串行计算阶段。


![图 1｜请求路径与指标。KV 是跨阶段状态；图中模型加载只适用于冷启动，热请求应单独计量。TPOT 的严格定义见下方时间戳公式。](images/01_request_path.png)

*图 1｜请求路径与指标。KV 是跨阶段状态；图中模型加载只适用于冷启动，热请求应单独计量。TPOT 的严格定义见下方时间戳公式。*


### 时间戳、阶段耗时与用户延迟

设客户端发出请求时刻为 `t0`，收到首个输出 token 为 `t1`，收到第 N 个输出 token 为 `tN`，收到完成事件为 `t_done`。在本文的客户端口径中：

$$
TTFT=t_1-t_0,\qquad TPOT=\frac{t_N-t_1}{N-1}\quad(N>1)
$$
$$
E2E=t_{done}-t_0,\qquad ITL_i=t_i-t_{i-1}\ (i\ge2)
$$
当 `t_done=tN` 且每个 token 单独可观测时，`E2E = TTFT + (N−1)×TPOT`。完成事件的额外收尾时间应单独解释。N≤1 时 TPOT 不适用，记为缺失值，不能伪造为 0。流式响应若一次发送多个 token，chunk 间隔不等于 token 间隔；服务端 decode 时间也不等于客户端 TPOT。

| 指标 | 回答的问题 | 主要关联 | 口径注意 |
|---|---|---|---|
| TTFT | 首 token 多久可见？ | 排队、输入准备、Prefill、首 token 选择与传输 | 不等于纯 Prefill kernel 时间 |
| TPOT / ITL | 后续生成是否流畅？ | Decode、KV 读取、调度、输出发送 | 请求均值与 token 加权均值不同 |
| Output throughput | 每秒生成多少输出 token？ | batching、生成策略、并发和调度 | 总输出 token / 统一测量窗口秒数 |
| Request throughput | 每秒完成多少请求？ | 请求长度与系统容量 | 不与 token/s 混用 |
| P50 / P95 / P99 | 典型与尾部请求有多慢？ | 队列、负载、抖动 | 必须注明是 TTFT、E2E 还是 ITL 的分位数 |
| Peak memory | 运行时最大显存占用？ | 权重、KV、激活、workspace、分配器 | allocated、reserved、进程显存不能混写 |
| 成功/超时/拒绝/OOM | 服务是否可用？ | 接纳、容量、资源故障 | 不能只平均成功请求而隐去失败 |

分位数要记录样本量和算法；少量 smoke 请求不足以稳定描述 P99。并发数表示同时在途请求数，batch 表示某次执行组织的请求数，两者并不相等。开放式到达流与固定并发的闭环测试也不是同一种压力条件。


### workload 决定瓶颈

| 类型 | 典型形态 | 优先指标 | 下一步 |
|---|---|---|---|
| 交互短请求 | 输入与输出都短 | TTFT、P99、超时率 | 队列、Prefill、启动/同步 |
| 长上下文 | 长输入或持续增长历史 | TTFT、Cache、峰值显存 | Attention、Chunked Prefill、KV |
| 批处理 | 可以等待，追求产出 | token/s、成本、完成率 | batch、kernel、量化 |
| 突发并发 | 到达率短时激增 | queue wait、P95/P99、拒绝率 | 接纳控制、调度、扩缩容 |

不能只用平均 Prompt 长度描述 workload；还应保留长度分布、请求顺序、共享前缀、输出长度、停止条件和服务目标。


### 每次实验至少保留四层证据

| 证据层 | 记录内容 | 用途 |
|---|---|---|
| 请求条件 | 模型/revision、tokenizer、prompt tokens、实际 generated tokens、batch、concurrency、到达模式、dtype、cache policy | 确认两次测量可比较 |
| 阶段表现 | Prefill/Decode 时间、TTFT、TPOT、ITL | 找到主要阶段 |
| 服务表现 | E2E、吞吐、P50/P95/P99、排队、成功率/拒绝率/超时率 | 判断用户可见收益 |
| 资源表现 | Peak memory、GPU 利用率、Cache 使用量、带宽和通信 | 解释收益来自计算、搬运还是容量 |


若 TTFT 变高但 Prefill kernel 时间没变，能认定 Attention 变慢吗？不能；应先检查排队、输入准备和首 token 发送。

现在已经知道要分别记录什么。先把排队因素固定，沿着第一个可重复现象往下问：输入越长，首 token 为什么越迟？


<a id="question-02"></a>

## 02｜为什么输入变长，会拖慢第一个 token？

Prefill 面对的是一整段已知输入。每个位置产生 Q、K、V，并按因果约束查询允许访问的历史位置。首 token 还没有生成，并不意味着模型只需要处理输入末尾一个 token：各层仍要建立输入上下文的表示和后续会用到的状态。

先只看 Attention 的核心乘法。对等长 Prefill，Query 和 Key 两个序列维度都随输入增长，完整 score 的形状含有 `S×S`；对带缓存的单步 Decode，Query 长度通常只有 1，Key 长度则包括历史。于是同一个公式在两个阶段呈现不同工作形态。输入从 4096 增至 8192，完整 score 的理论元素数变成四倍；这不是端到端 TTFT 必定四倍，因为投影、其他层、访存和执行路径也参与耗时。


### Attention 计算与张量形状

$$
Attention(Q,K,V)=Softmax(QK^\top/\sqrt D+M)V
$$
令 B 为 batch，H 为 Query heads，Hkv 为 KV heads，D 为 head_dim；Sq 为当前 query 长度，Sk 为含历史 Cache 的 key 长度：

| 对象 | 形状 | 说明 |
|---|---|---|
| Q | `[B,H,Sq,D]` | 当前要查询的位置 |
| K / V 与 Cache | `[B,Hkv,Sk,D]` | Cache 保留未扩展的 KV heads |
| score / probability | `[B,H,Sq,Sk]` | Prefill 常有 Sq=Sk，单步 Decode 常有 Sq=1 |
| Attention 输出 | `[B,H,Sq,D]` | 再转置、合并成 `[B,Sq,H×D]` 并做输出投影 |

MHA：Hkv=H；GQA：1<Hkv<H，教学实现要求 H 能被 Hkv 整除；MQA：Hkv=1。RoPE 作用于 Q/K 的位置表示，位置偏移也影响 Cache 兼容性。

实现顺序是 投影 → reshape/transpose → 在序列维拼接历史 K/V → 对齐 KV head → score/mask → softmax → 聚合 V → 合并 head/输出投影。必须先保存较少的 KV heads，再临时对齐；否则会失去 GQA 的缓存容量收益。真实 kernel 可以直接处理分组头，不一定物化 `repeat_kv` 的拷贝。

正确性至少检查非法 head 配置、mask 广播、有限输出，以及“全序列最后一个位置”与“前缀 Cache + 单步 Decode”在容差内一致。带历史的多 token 输入要按全局位置构造因果 mask，不能机械使用局部方阵 mask。


Prefill 的 `Sq=Sk=S` 与 Decode 的 `Sq=1` 是工作量分析的起点，但 causal mask、分组头和真实 kernel 都会影响执行。

张量变大解释了“工作增加”，尚未解释“时间花在哪里”。下一问需要把计算量与搬运量分开。


<a id="question-03"></a>

## 03｜工作变多，就一定是 GPU 算力不够吗？

假设一个 kernel 的计算单元一直在等数据，即使提高理论峰值算力，也未必缩短等待；另一种情况是数据已经就位，但矩阵乘法饱和，这时减少搬运未必是主要收益。还有一种更早发生的问题：状态根本放不进显存。

所以先依次回答：能否装下？数据是否来得及搬？计算单元是否忙满？Roofline 将后两问连接起来：同样多的计算，如果反复读取中间结果，每搬运一个字节完成的计算更少，就更容易受带宽约束。片上存储的价值在于复用，但片上空间本身也有限。


### 先分开“装不下”“搬得慢”“算得慢”

显存容量限制决定是否 OOM；显存带宽限制决定数据能否及时到达计算单元；计算吞吐限制决定运算本身多快。峰值算力不是程序速度。SM 组织线程块与片上资源，warp 是线程执行组织单位；CUDA Core 服务通用计算，Tensor Core 服务受 dtype 和形状条件约束的矩阵乘加。

寄存器 → Shared Memory（片上 SRAM）→ L2 → 全局显存，是观察数据驻留与复用的层次。容量、延迟、带宽和作用域不同，不能用单一数字排序所有代价。连续线程访问连续地址有利于合并访存；寄存器和 shared memory 占用过高可能降低 occupancy。

$$
I=\frac{FLOPs}{Bytes},\qquad P_{roof}=\min(P_{compute}, BW\times I)
$$
Roofline 给出定位假设，不是 profiler 实测；Bytes 必须声明是输入输出理论体积，还是实际 HBM 流量。矩阵乘法 `[m,k]×[k,n]` 的主要 FLOPs 约为 `2mnk`。内存带宽代理值、GPU 代际对照仅用于教学，不能当成本机规格。


### SRAM 优化的四个联动因素

| 因素 | 希望得到什么 | 可能抵消收益的成本 | 教学小验证 |
|---|---|---|---|
| 数据复用 | 一次搬入、多次使用 | 搬入和同步本身需要时间 | `sram_gain` |
| bank 布局 | 并行访问不同 bank | stride、padding、对齐造成冲突 | `bank_access_report` |
| tile / layout / occupancy | 工作集可驻留且并行度够 | tile 过大占用资源，过小复用不足 | `layout_tile_score` |
| 寄存器预算 | 临时状态留在近处 | spill 重新产生较慢访存 | `spill_tradeoff` |

同一地址的 broadcast 读取不能按“多个不同地址映射同一 bank”的普通冲突计算。教学 score/net_gain 不是真实时间，也不能直接计算 occupancy。


显存剩余很多不能排除带宽瓶颈；显存接近满也不能单独证明 kernel 受带宽限制。

回到刚才的 `S×S` score：如果大量时间花在它的写回和读入，能否保留 Attention 的结果，却不把完整中间矩阵存下来？


<a id="question-04"></a>

## 04｜不保存完整 Attention 矩阵，结果还能一样吗？

先尝试最自然的方案：把 Key/Value 分块，每次只计算一个 score tile。困难马上出现——softmax 的分母覆盖整行 Key，如果每块各自归一化再相加，各块会获得错误的相对权重。

用一行两项分数 `[0, ln(3)]` 和两个标量 Value `[0, 1]` 看这个问题：整行 softmax 是 `[1/4, 3/4]`，输出应为 `3/4`。若每块只有一项，独立 softmax 都等于 1，直接相加得到 1，简单平均又得到 1/2，都不正确。

解决方法不是放弃分块，而是携带足够的跨块统计量：当前最大分数、指数和、未归一化的加权和。新块提高最大值时，旧统计量必须同步缩放到新的数值基准。这样才能既保持全局归一化，又避免物化完整矩阵。


### 完整 score 与 tile 工作集

标准非分块 score 的理论元素数为 `B×H×Sq×Sk`；Prefill 等长情形为 `B×H×S²`。单个 score tile 为 `Bq×Bk`；一维 tile 数是 `ceil(S/b)`，二维未考虑 causal 剪枝的 tile 数是其平方。**单 tile 的体积、整个算法总工作量和真实 HBM 流量是三个不同量。**

FlashAttention 避免把完整 score/probability 矩阵反复写回 HBM，仍执行所需的 Attention 计算。tile 越大，循环数可能越少，但寄存器/shared memory 压力越高；tile 越小，片上压力越低，但循环与调度成本增加。不能由工作集压缩倍数直接推出加速倍数。


### Online Softmax 的合并公式

对固定的一行 Query，维护最大值 m、指数和 l、**未归一化**加权和 u。新 tile 的分数为 s，值为 Vtile：

$$
m'=\max(m,\max(s)),\qquad a=\exp(m-m'),\qquad p=\exp(s-m')
$$
$$
l'=a l+\sum p,\qquad u'=a u+pV_{tile},\qquad O=u'/l'
$$
初始 `m=−∞, l=0, u=0`。若维护的 O 已归一化，则更新为 `(a×l×O+pVtile)/l'`。不能对每个 tile 独立 softmax 后直接相加，因为归一化分母覆盖整行 Key。

分块 Attention 的 CPU 教学实现采用 Q/K/V 双层分块循环，涵盖 causal、非整除尾块、float64、大 score 稳定性与非法输入；其输入是二维 `[seq_len,dim]`，未实现真实 CUDA/Triton kernel、batch/head、dropout 或 backward。真实累加器可能采用更高精度。推广到任意 mask 时，还要显式处理整行都被屏蔽导致的零分母/非有限状态。


### API、dispatch 和服务结果

SDPA 是表达 Attention 的 API；具体 kernel 由设备、dtype、shape、mask、版本等条件决定。调用 SDPA 不足以证明用了 FlashAttention。验证顺序为：**语义正确 → 实际 dispatch 可确认 → kernel 证据可解释 → 请求指标可复测**。

可以按减少写回、提高并行效率、匹配新硬件理解 FlashAttention 及后续版本；这些是阅读线索，具体版本能力仍要在目标环境确认。Prefill 优化不能直接推出 Decode TPOT 改善。

| 信号 | 需要检查 | 可探索方向 |
|---|---|---|
| 计算单元接近饱和 | FLOPs、dtype、矩阵形状、Tensor Core 路径 | kernel、融合、精度或形状 |
| 带宽接近上限 | HBM 流量、重复读写和复用 | tiling、FlashAttention |
| kernel 很短但空隙多 | launch、同步、CPU 准备 | 融合、batch、Graph/编译 |
| 长度变化时抖动 | padding、bucket、动态 shape | 分桶与稳定执行形状 |


上述例子处理第二块时，最大值从 0 变为 `ln(3)`，旧分母缩为 `1/3`，新分母为 `4/3`，加权和为 1，所以结果仍是 `3/4`。

至此解决的是一次 Attention 如何执行。即便首 token 更快，后续 127 个 token 仍要逐步生成：为什么它们不能同样一次算完，又为什么需要保存历史？


<a id="question-05"></a>

## 05｜有了 KV Cache，后续生成为什么仍有成本？

自回归生成的下一步输入依赖刚选出的 token。在普通逐步生成中，后面的 token 尚未确定，不能像已知 Prompt 那样全部并行处理。KV Cache 避免重复计算历史位置的 K/V，但新的 Query 仍要利用历史 K/V 完成 Attention；缓存省掉了重复计算，同时引入持续驻留和读取的状态。

把成本分成两本账更清楚：**容量账**问当前保存多少字节；**每步时间账**问读取权重、读取 KV、执行算子和等待调度各用了多久。短上下文、低 batch 的 Decode 可能更受权重读取影响；上下文增长后，KV 读取和容量也会越来越重要，不能一见 TPOT 高就断定只有 KV 是原因。

对贯穿全文的 4096-token 请求，下面的模型参数给出 512 MiB 的理论 KV。若输出全部继续写入缓存，在 4224 个已缓存位置时对应 528 MiB。这里只算状态规模；最终已缓存位置是否包含最后一个输出 token，要看停止时机和实现。


### 先算清字节账本

对于常规 K/V 等头维、每层形状一致的 Cache：

$$
M_{KV}=2\,L\,B\,H_{kv}\,D\,S\,b
$$
其中 L=层数，B=batch，Hkv=KV heads，D=head_dim，S=缓存 token 数，b=每元素字节数；2 表示 K/V 两份。**计算字节数时必须计入每元素字节数 b。**变长请求应将 B×S 改为各请求实际长度之和；共享页、量化元数据与特殊架构需另算。

例如 L=32、Hkv=8、D=128、b=2，每个 token 的 KV 为 131072 bytes=128 KiB，4096 token、batch=1 为 512 MiB。其余不变时，Hkv=32 为 2 GiB，Hkv=1 为 64 MiB。这些是理论 KV 容量，不是进程 peak memory。

$$
M_{peak}\approx M_{weights}+M_{KV}+M_{activations}+M_{workspace}+M_{allocator/other}
$$
这个加法是容量分解，不代表各项独立峰值可以直接相加；实际 peak 取决于张量生命周期。GB=10⁹ bytes，GiB=2³⁰ bytes，要明确区分。

| 层次 | 改变对象 | 代表机制 | 不应混淆 |
|---|---|---|---|
| 表示 | 每 token 保存或重构的状态 | MHA/GQA/MQA/MLA、KV 量化 | 与分页不是一件事 |
| 分配 | 逻辑状态映射到物理空间 | PagedAttention | 不自动消除每 token 的状态 |
| 复用 | 已有上下文是否重新计算 | Prefix/Radix Cache | 不等于只改变物理布局 |
| 治理 | 状态保留多久、何时回收 | LRU/价值评分、驱逐、重算 | 不等于请求优先级 |

MLA 的教学估算 `L×B×S×latent_dim×b` 只是潜在表示代理；真实 MLA 可能还需额外位置状态和恢复计算，不能据此宣称完整架构压缩比。


![图 2｜表示、分配、复用与治理。两组共享 KV 是 GQA 示意；MQA 只有一组。文字块只作 token 路径示意，不规定 tokenizer 的实际切分。](images/02_kv_layers.png)

*图 2｜表示、分配、复用与治理。两组共享 KV 是 GQA 示意；MQA 只有一组。文字块只作 token 路径示意，不规定 tokenizer 的实际切分。*


GQA 减少 KV heads；FlashAttention 改变计算与访存路径。前者改变状态规模，后者不会自动把同一模型的 KV heads 变少。

单请求的字节数算清了，真实服务却有长短不一、随时增长和结束的请求。总容量够用时，为什么还需要分页？


<a id="question-06"></a>

## 06｜显存总量够，为什么还要把 Cache 分页？

如果每个请求都按最大长度预留一大片连续空间，短请求会占着暂时用不到的位置，增长请求又可能找不到合适的连续区域。分页把“逻辑上连续的 token 历史”与“物理上放在哪些块”分开，让请求按需领取块，并在结束后归还。

例如每块放 16 token，长度 17 需要两块，共 32 个槽位，尾部空 15 个。增长到 18 仍是两块；从 16 增到 17 才需要新增一块。分页减少大段预留的浪费，但尾块仍可能有内部碎片，块表和管理也有成本。

这里的核心不是一个取整公式，而是状态一致性：若新块不足，不能先改长度再发现分配失败；若请求结束，不能释放仍被别的请求共享的块。


### PagedAttention 的账本与原子性

block_size=P 时，长度 S 需要 `ceil(S/P)` 块；分配槽位为 `P×ceil(S/P)`，尾块浪费是分配槽位减 S。第 t 个逻辑 token 位于逻辑块 `t//P`，块内偏移 `t%P`，再通过 block_table 找到物理块。

```text
逻辑 token → 逻辑块号 / 块内偏移 → block_table → 非连续物理 KV 块
Prefill: 初始分配 → Decode: 跨边界扩容 → Finish: 释放 → 新请求复用
```

分页管理的关键不变量：空闲池与已分配集合互斥；块索引合法；逻辑长度与块表一致；OOM 不留下半分配；释放不重复；恢复逻辑 Cache 必须按块表顺序，不能按物理地址排序。教学恢复会拼接物理块，真实 kernel 可以直接按映射读取。

通用扩容判断应比较新旧 `ceil(length/P)`。`new_len % P == 1` 仅适用于特定单 token、P>1 的边界直觉，不是覆盖 P=1 和多 token 追加的通用规则。共享物理页需要引用计数/写时复制等额外语义，不属于此独占块教学管理器。


块从哪里来、属于谁、何时归还，必须形成守恒账本。分页不会让相同 token 的 K/V 自动只算一次。

既然块已经可以独立管理，两个请求如果以同样的系统提示开头，能否让它们共享已经算好的前缀？


<a id="question-07"></a>

## 07｜相同的开头能复用吗，缓存又该留多久？

设两个请求都以 3072-token 的相同前缀开始，各自总长 4096 token。如果模型和上下文兼容、backend 支持该命中粒度，第二个请求可以复用前缀状态，只对剩下的 1024 token 做新增 Prefill。但新增 Query 仍需关注被复用的历史 Key；因此“少算 75% 的输入位置”不能直接等同于“TTFT 降低 75%”。

接着问什么才算“相同”：当前一块 token 相同还不够，因为它的表示依赖之前的上下文。缓存了 A→B 和 C→D，不能拼成 A→D。前缀树或父块哈希链要保留这种路径身份，模型版本、位置编码、adapter 和访问隔离也必须兼容。

再往深处一层：能复用不代表值得永久保留。留下一段冷前缀会占用活跃请求所需的空间。于是匹配问题自然变成保留、驱逐、重算和压缩之间的资源决策。


### Radix 匹配与可复用状态

Radix Tree 用压缩边保存 token 路径。插入时计算最长公共前缀，部分重合则分裂成 shared、旧后缀、新后缀，并保留旧分支的子节点、terminal 和缓存指针。匹配返回 `hit_len`，再拆成 `hit_prefix` 与 `miss_suffix`；只有后者需要新增 Prefill。

terminal-only Radix 教学实现只把有完整 terminal 路径的命中计入可复用长度，**不能把它等同于任意 token 级 LCP 或真实 backend 的全部命中能力**。完整命中后，如何取得下一个 logits、是否需重算末 token，也要看实际 backend。

前缀相同需要是 token 与上下文兼容，而非仅文本看起来相同：模型 revision、tokenizer/特殊 token、位置/RoPE、adapter、量化表示及租户隔离都要确认。采样参数通常不改变已固定 token 前缀的 K/V 本身，但 RNG、生成控制状态和服务缓存键是否兼容要分别检查，不能笼统认为改 temperature 就必然改变 KV。


### 从完整 block 命中到 LRU

完整块前缀缓存的教学模型只匹配从位置 0 开始的连续完整 block，首个 miss 停止，尾部不足一块不参与。每次写入/访问刷新逻辑时钟，容量超限时驱逐最旧项。

请求命中率 `hit_requests/requests` 与 token 复用率 `reused_tokens/total_prompt_tokens` 不同；复用 token 占比也不等于实际 Prefill 时间下降比例。



### 价值驱逐与生命周期

缓存价值调度维护 prefix、大小、命中次数、最近访问、当前占用。保留分数由复用奖励、最近访问奖励和容量惩罚组成；`recency=1/(1+time_gap)`。低分优先驱逐，heap 更新留下历史记录，弹出时要与当前 entry 比较并跳过 stale 项。

始终检查 `current_bytes = sum(entry.size)`，拒绝单条超过总容量的状态。

| 动作 | 收益 | 代价 | 证据 |
|---|---|---|---|
| 保留 | 后续可复用 | 压低可接纳并发 | 占用、命中率 |
| 驱逐 | 释放容量 | 再次 Prefill | 驱逐、重算 token、TTFT |
| 重算 | 少驻留状态 | 多计算与等待 | recompute 时间 |
| 压缩/量化 | 每 token 字节减少 | 误差与恢复/读写代价 | 字节、质量、kernel、延迟 |

完成、异常取消、版本失效都需要正确清理引用。TTFT 下降不能单独证明命中，必须有 backend metrics、日志或接口返回的命中证据。


一个短前缀命中的请求和一个长前缀命中的请求，都能让请求命中率增加，但节省的工作不同。应同时记录复用 token、维护成本和实际延迟。

分页解决分配，前缀复用减少重复工作，驱逐回收冷状态。若活跃权重与状态本身仍太大，就需要进一步问：每个数能否少占一些字节？


<a id="question-08"></a>

## 08｜把数值压成低比特，为什么不一定更快？

量化先是一种数值表示变化：用有限编码近似原数值，读取时依靠 scale 和可能的 zero-point 解释。它同时改变存储体积、误差和执行成本，这三件事需要分别证明。

例如步长为 0.1、零点为 0，数值 0.26 在不发生裁剪时映射为整数 3，恢复为 0.3。误差来自舍入；若数值超过可表示范围，还会出现裁剪误差。缩小分组可能更贴合局部分布，却增加 scale 数量和读取处理成本。

然后区分压缩对象：权重通常跨请求复用，KV 随请求增长。压权重可能解决模型装载，压 KV 可能扩大长上下文并发；二者对时间和质量的作用不同。最后才问执行路径：若每次前向先展开完整浮点权重，再调用浮点矩阵乘法，文件变小并不意味着计算已经变快。


### 统一两种 scale 约定

常见步长写法：

$$
q=clip(round(x/\Delta)+z,q_{min},q_{max}),\qquad \hat x=\Delta(q-z)
$$
对称量化 z=0，非对称量化允许零点偏移。步长形式使用 Δ；倒数形式使用 `s=1/Δ`，即 `q=round(x×s)`、`x_hat=q/s`。二者不能交叉反量化。全零张量要有安全 scale，round/clip 引入表示误差。

per-tensor 元数据少但容易受局部离群值影响；per-channel 适应通道差异；group-wise 在误差与 scale 数量之间折中。分组数向上取整，最后不足一组不能丢失。激活随 token、层和上下文变化，校准输入要覆盖实际场景；SmoothQuant 是权重与激活分布协同处理的一个方向。

7B 参数的理想权重体积：FP16 14 GB、INT8 7 GB、packed INT4 3.5 GB；未计 scale、zero-point、bias、未量化层、临时反量化张量、KV 和分配器。**用 int8 张量保存 4-bit 数值范围仍占 1 byte/元素，不是 packed INT4。**


### 对象、时机、格式分开看

| 路线 | 对象/时机 | 首先验证 | 主要风险 |
|---|---|---|---|
| PTQ | 训练后量化 | 校准、误差和可加载性 | 数据不代表服务分布 |
| QAT | 训练/微调时感知量化误差 | 质量恢复与训练成本 | 数据、时间与算力投入 |
| W8A16 | 8-bit 权重、16-bit 激活 | 存储与前向输出 | 反量化可抵消速度收益 |
| GPTQ / AWQ | 部署前权重校准 | 真实 artifact、loader、kernel | 教学模拟不等于算法实现 |
| FP8 | 权重/激活/计算或 Cache 路径 | 格式、scale、硬件和累加精度 | 8-bit 不代表相同数值语义 |
| KV 量化 | 请求执行时持续保存状态 | 容量、长上下文质量、读取代价 | 误差随服务使用方式体现 |
| GGUF | 文件格式与部署封装 | 匹配 loader/backend，如 llama.cpp | 不是 GPTQ/AWQ 一类算法 |

PTQ 满足质量门槛就先验证部署；PTQ 质量损失过大且有代表性训练数据/预算时，才进一步考虑 QAT。低 bit 不必然更快、更准确或更适合服务。


### W8A16 的前向

W8A16 实现以 `s=127/absmax` 映射，存储范围为 int8；前向把 `weight_int8` 转为激活 dtype，再除以 scale，使用浮点 `F.linear`。bias 也要匹配 dtype。量化权重与 scale 作为 buffer 随状态和设备迁移。

因此这里证明的是低比特**保存**与近似浮点**计算**的关系，不是原生 INT8 GEMM 性能。还原完整浮点权重会产生临时显存。单层权重缩小不等于完整模型 peak 减半。测试包括全零、范围、存储账本、显式反量化参考、FP16/FP32 输出契约。


### GPTQ/AWQ：校准统计与教学简化

分组权重量化的教学模拟器从输入激活最后一维汇总 RMS 重要性；按输入通道分组，每个输出行/组保存 scale。AWQ 分支挑高重要性通道，用浮点副本保护，其他权重量化；恢复时按组反量化并回填保护位置。

**这里的 `gptq` 分支是分组 absmax 模拟，没有实现完整 GPTQ 的二阶误差补偿；`awq` 分支的浮点保护也不等于真实 AWQ 的完整缩放/搜索与部署算法。**两者的名字用于教学区分，不能把误差对比解释成真实 GPTQ vs AWQ benchmark。保护副本/mask 还会产生额外存储。

可选 GPU 实验从真实模型 q_proj hook 采集激活，但证据仍是 `gpu_simulation_on_real_model_state`。成熟库 artifact 需另测；权重 MSE、校准输出 MSE 不替代任务准确率、格式或工具调用成功率。


### FP8 与 KV 分组量化

真实 FP8 的 E4M3/E5M2 在指数范围和尾数精度上不同；并非均匀 int8 量化。教学模拟器 `FP8KVCacheSim` 实际用 INT8 容器演示 scale 与恢复，不能称为真实 FP8 Tensor Core 测试。

KV 按最后一维分组，保存 q、各组 scale 和原始 shape；恢复必须使用同一位置同一组的 scale。group 更小通常提高局部适应性，但元数据与处理次数增加。真实存储应计 `q bytes + scales bytes + 其他状态`。KV 状态实验从 past_key_values 取得真实 KV 后再模拟，证据是 `gpu_simulation_on_real_kv_state`；torchao FP8 weight-only API 探针只测权重路径，不是 KV Cache 证据。


### 部署验证闭环

**Artifact → loader/backend → 实际 kernel → 固定 workload → 质量与服务决策**。

分别检查数值有限性/shape/logits，生成长度/停止/重复率，目标任务与长上下文质量，最后比较 TTFT、TPOT、吞吐、P99、peak memory。格式加载成功只证明装载可行；显存下降但速度不升时，检查反量化、隐式回退、kernel 和 workload。


INT4 数值装在 INT8 容器里不会自动变成半字节存储；能够加载量化文件，也不证明走了匹配的高效 kernel。

到这里，我们一直在减少每步需要保存或搬运的数据。还有一个正交方向：减少生成固定长度回答所需的昂贵步骤。先明确每一步究竟怎样从分数选出 token。


<a id="question-09"></a>

## 09｜模型给出 logits 后，下一个 token 是怎么选出的？

模型输出的是词表上的分数，并不是唯一确定的文字。greedy 选最大项，采样从处理后的分布抽取；temperature、Top-K、Top-p 改变候选分布，因此也会影响质量、输出长度和后续路径。

先用排序理解 temperature：正温度除法不改变 logits 的相对顺序，却改变 softmax 后的概率。再看筛选：Top-K 限制高分候选集合，Top-p 按概率累积保留足够的质量。两者连续使用时，后一步面对的已经不是原始分布。

这一步之所以要先讲清楚，是因为下一问的加速不能只看“候选字面上是否一致”。如果目标是保留目标模型的采样分布，接受和拒绝就必须依赖实际的概率。


### 从 logits 到 token

本节采用的顺序是 **Temperature → Top-K → Top-p → Softmax → 选择**。温度要求有限且为正；`logits/T` 改变分布尖锐程度，不改变排序。greedy 使用 argmax，不能把“小但非零温度的采样”当作 greedy。backend 的 `temperature=0` 可是禁用采样的接口约定，不等于把 0 传给本教程的除法函数。

Top-K 以第 K 大分数作阈值时，同分候选可能超过 K。Top-p 在当前候选分布中排序、累计，并保留首次达到/跨过 p 的边界 token；恢复原词表顺序后，候选外设为 `−∞`，重新 softmax。

例：概率 `[0.5,0.3,0.1,0.05,0.05]`，p=0.85 时保留前三个，重归一化分母是 0.9。Top-K 与 Top-p 的先后会改变分布。边界等号的具体实现应记录；不能混用不同实现而声称采样配置完全相同。

生成循环还需要记录 EOS、最大输出长度、停止原因、实际输出 token 数和随机种子。测试关注 shape、dtype/device、非法温度、ties、Top-p 边界、batch、选择与追加循环。


固定随机种子有助于复现，却不能弥补不同采样流程造成的分布差异；停止条件不同也会让 token/s 比较失去共同基础。

现在可以提出更大胆的方案：让便宜的路径先猜多个 token，再让昂贵的目标模型集中验证，能否减少等待？


<a id="question-10"></a>

## 10｜能否一次验证多个 token，而不改变目标分布？

一个普通 Decode 步骤推进一个 token。投机解码尝试让草稿先提出一条候选路径，目标模型利用这条已知路径集中计算验证概率；但被接受的输出必须是连续前缀，不能跳过一个被拒绝的位置继续沿用它后面的上下文。

接受规则把草稿分布 q 与目标分布 p 联系起来：草稿提出得过于频繁的候选不能全部接受，拒绝时还需要修正分布补回概率。全部接受时，则可利用额外的目标位置产生 bonus。公式成立的前提是对齐正确的上下文与实际采样分布。

假设 target-only 推进 4 token 用 80 ms，某投机轮草稿加验证等总计 50 ms。如果该轮实际提交 4 token，才有可比较的时间收益；若只提交 1 token，就是更慢。这里的数字仅解释盈亏口径，不是硬件实测。


![图 3｜拒绝后修正与全接受后 bonus。图示的是接受判定的先后；target 概率可集中计算，不能据此理解为必须逐个 target forward。](images/03_speculative.png)

*图 3｜拒绝后修正与全接受后 bonus。图示的是接受判定的先后；target 概率可集中计算，不能据此理解为必须逐个 target forward。*


### 经典 draft/target 投机解码

草稿模型按上下文提出 K 个候选，目标模型对候选路径集中验证。`draft_probs` 为 `[K,Vocab]`，`target_probs` 为 `[K+1,Vocab]`，额外一行提供全部接受时的 bonus。

令 p 为 target、q 为 draft。对于已按 q 采样的候选 x：

$$
\alpha(x)=\min(1,p(x)/q(x))
$$
逐位置接受，首个拒绝后结束本轮，并从修正分布采样：

$$
r(x)=\frac{\max(p(x)-q(x),0)}{\sum_y\max(p(y)-q(y),0)}
$$
全部 K 个候选通过后，从目标模型额外位置采样 bonus token。因此本轮输出是“连续接受前缀 + correction”或“K 个接受 token + bonus”。后续偶然相同的 token 不能跨过首次拒绝继续算接受。实际 EOS/长度预算还可能提前截断。

概率形状、归一化、有限性、非负性与 token 索引必须验证。q(x)=0 的输入边界与 residual 分母也应有明确约定。上述分布保持语义依赖正确的概率对齐、采样和回退；不能拿逐 token 相等检查代替概率验证算法。


### 从接受率到净收益

$$
acceptance=\frac{accepted}{proposed},\qquad
effective\ progress=\frac{committed\ output\ tokens}{rounds}
$$
`accepted_tokens/rounds` 不包括 correction/bonus，和有效输出推进量不是同一指标。一个粗略盈亏条件是：候选阶段、验证、修正和额外调度的总时间，小于 target-only 生成相同有效输出所需时间；高接受率本身不足以证明加速。

| 分支 | 候选来源 | 必须新增的证据 |
|---|---|---|
| 经典 draft/target | 小模型自回归草稿 | acceptance、target calls、draft/verify cost |
| multi-token / 多头 | 一次多个候选 token 或预测头 | 首拒绝位置、有效推进、验证预算 |
| self-speculative / early exit | 同模型浅层或内部路径 | 草稿代价、恢复/回退语义 |
| tree / multi-candidate | 多条候选路径 | 树验证成本、提交路径、容量 |
| DFlash（扩展方向） | block diffusion 草稿 | block 长度、接受前缀、draft/verify cost |
| DSpark（扩展方向） | 半自回归与置信度预算 | confidence、proposal budget、负载下 TPOT |

DFlash/DSpark 作为扩展方向介绍，本次没有独立验证其最新实现。单请求候选预算与服务调度会互相影响：候选变多也可能加重 batch 和验证成本。

连续前缀匹配的 CPU benchmark 是给定 draft/target token 序列上的连续相等前缀模拟，`effective_output_tokens` 由接受与修正组成；它与 经典投机解码的概率接受、residual、bonus 机制不是同一算法口径。真实 benchmark 先固定质量，再看吞吐、TTFT/TPOT、接受率、目标调用、draft/verify 代价、回退次数及额外显存。


接受率描述草稿适配程度，有效推进量描述真正提交多少输出，端到端时间才回答是否加速。三者不能互相替代。

单请求层面的路线已有计算、缓存、量化和投机。把很多请求同时放进服务后，为什么这些局部收益还可能伴随更差的用户体验？


<a id="question-11"></a>

## 11｜单请求快了，为什么多人同时用反而更慢？

现在把教学请求从一个增加为多个。请求进入系统不等于立刻获得 GPU：调度器要判断谁 ready、谁有足够 Cache、谁进入下一批。一个请求等待别人的执行，就会增加客户端延迟，却不一定增加自己的 kernel 耗时。

Batch 可以让一次执行服务多个请求，提高资源利用，但每个请求可能等待组批或下一轮。吞吐回答系统每秒完成多少工作，TTFT/TPOT 回答某个用户等待多久，所以两者方向可以相反。

最简单的优先级排序能说明“下一步选谁”，但持续到达的高优先级请求可能让其他请求饥饿。想证明调度改善，除了平均速度，还必须看请求等待、完成率和尾部。


### 调度范围逐层扩大

| 层次 | 决策 | 输入 | 结果 |
|---|---|---|---|
| 单步 Decode | 谁得到下一轮推进？ | 阶段、优先级、剩余工作 | 顺序、等待步数、公平性 |
| Cache 资源 | 谁可以驻留/暂停/恢复？ | 空闲容量、块、热度 | 可接纳并发与 OOM 风险 |
| batch/队列 | 哪些 ready 请求组成下一批？ | 到达时间、Prefill/Decode 预算 | 吞吐、queue wait、尾延迟 |
| 服务池 | Prefill/Decode 由谁执行？ | 实例、GPU、服务等级 | 隔离、池利用率、交接 |
| 异构路由扩展 | 传 KV、重算还是同池？ | 状态大小、链路、两池能力、SLO | 路由、回退与成本 |

这里将调度划分为四个范围，加上异构跨池路由扩展。vLLM 和 SGLang 都是完整服务 backend，不能把前者限定为分页、后者限定为前缀树；跨 backend 性能差异是服务栈整体结果。


### 单步请求状态与静态排序的局限

单步调度教学模型的状态是 Prefill → Decode → 完成。入队保存输入长度、生成上限、priority、cache hit；每次只推进选中的一个请求，未选中的活动请求累计 wait_steps。

排序键为 `(phase_rank, cache_rank, -priority, total_len, request_id)`，教程中 Prefill 优先、cache hit 优先，其余字段逐级稳定打破平局。该规则可用于观察顺序，但不保证生产公平性：持续到来的高优先级/Prefill 请求可能压住其他请求。`max_steps` 是循环保护，不是公平机制；wait_steps 不是毫秒。


并发为 16 不等于某次 GPU 执行一定包含 16 个请求；Cache 不足、到达时刻和阶段预算都可能改变实际 batch。

一个具体冲突随之出现：长 Prompt 的 Prefill 占据执行机会时，正在逐字输出的请求能不能获得更及时的推进？


<a id="question-12"></a>

## 12｜长输入怎样给正在生成的请求让出执行机会？

先区分两个边界：请求边界与迭代边界。若必须等整批请求全部结束才能换人，短请求完成后留下的机会难以及时利用；Continuous Batching 在迭代边界重新组织 ready 请求。若一次长 Prefill 仍持续太久，就再把未命中的输入后缀切成 chunk，在 chunk 边界让 Decode 有机会进入。

沿用前面 4096-token 输入、命中 3072-token 前缀的例子，剩余 1024 token 若按 256 切分，需要四个 chunk。Prefix Cache 决定还剩多少输入位置，Chunked Prefill 决定怎样安排这些工作，Attention kernel 决定每份工作怎样执行。三个层次可以配合，但不应归为同一种优化。

chunk 越小，让出机会越频繁；代价是调度、启动和更小矩阵可能降低效率。目标不是把 chunk 无限缩小，而是在给定负载下找到交互延迟与总体产出的可接受平衡。


### Continuous Batching 与 Chunked Prefill

Continuous Batching 在迭代边界重新查看 ready 集合，允许完成请求退出、新请求加入。Chunked Prefill 只切分未命中 suffix，保持顺序与尾块，并在 chunk 边界给其他 Decode 请求准入机会。

小 chunk 提高让出频率，但可能增加调度/launch 代价；大 chunk 有利于一次处理更多计算，但可能伤害 Decode 流畅度。它与 Prefix Cache 互补：前者决定**怎样处理剩余工作**，后者决定**剩余工作有多少**。


### 三种优化解决不同问题

| 机制 | 改变什么 | 对照 | 必须观察 |
|---|---|---|---|
| FlashAttention | Attention 中间结果的计算/读写路径 | 标准 Attention vs 分块 Attention | 输出一致性、kernel 时间、工作集/流量、TTFT |
| Chunked Prefill | 长输入进入调度器的粒度 | 完整 Prefill vs chunks | Decode 抖动、TTFT、TPOT、P99 |
| Prefix Cache | 已算过的相同前缀是否重复计算 | cache off vs 命中 | hit_len/reused_tokens、TTFT、Cache 占用 |

Prefill 中，Q/K/V 投影和输出投影带来矩阵计算；score、softmax 与 Value 聚合带来工作集与数据搬运；K/V 在计算中产生并写入 Cache。Cache 写入不是必须等到输出投影之后才发生，流程图表示依赖关系而非严格 kernel 排序。


把输入切块不会自动减少输入 token 数，也不保证总 Prefill 更快。必须观察 Decode 的 ITL/TPOT 是否变稳，以及 Prefill TTFT 付出了什么代价。

如果两类工作长期竞争，同池调度始终难以兼顾，下一步才是考虑让 Prefill 与 Decode 使用不同资源。拆开会带来什么新成本？


<a id="question-13"></a>

## 13｜拆开 Prefill 与 Decode，省下的时间会被传输吃掉吗？

分池的动机是隔离两种不同形态的工作，但 Decode 仍需要 Prefill 建立的状态。原来在同一设备上直接读取的数据，现在可能要跨设备搬运、转换布局并等待同步。拆分把竞争问题变成了竞争与交接的联合问题。

例如前面理论 KV 为 512 MiB，换成十进制约 536.87 MB。假设有效链路可达 100 Gbit/s，仅按大小除以带宽估算就约需 42.95 ms，还未包含固定开销和排队。只有隔离与资源匹配带来的收益足以覆盖暴露在关键路径上的交接成本，拆池才值得。

因此决策有三条路径：兼容且传得动就传 KV；传输代价过高时比较在目标池重算；隔离收益不大时留在同池。增加 GPU 后还要看并行通信和 MoE 专家负载，不能把更多硬件当作无成本加速。


![图 4｜同池分块与 PD 的关系。时间段不按实测比例绘制；同池回退是替代跨池路径，并非跨池传输完成后的必经步骤。](images/04_serving_pd.png)

*图 4｜同池分块与 PD 的关系。时间段不按实测比例绘制；同池回退是替代跨池路径，并非跨池传输完成后的必经步骤。*


### PD 与异构交接

长输入短生成倾向 Prefill 压力，短输入长生成倾向 Decode/KV 压力，两者都长是混合压力。逻辑分池要求每个请求只进入一个计划池，不能遗漏或重复；逻辑账本本身不证明真实 worker 已拆开。

分池的教学决策要求吞吐提高、P95 不恶化且 handoff 在预算内；实际服务还要同时看质量、失败、P99、资源成本和池利用率。异构路由将选择扩展为：

| 方案 | 何时有意义 | 需要付出/验证 |
|---|---|---|
| 传输 KV | 状态兼容且链路足够快 | transfer bytes、handoff、拥塞与等待 |
| 重算 Prefill | 传输太慢或状态不兼容 | 目标池的额外计算与队列 |
| 保持同池 | 隔离收益小于交接成本 | 原池竞争与尾延迟 |

用十进制 MB 与 Gbit/s，理想传输毫秒数为 `8×MB/Gbit_s`；若带宽是 GB/s，则是 `MB/GB_s`。GB/s 与参数 `link_gbps` 表示的 Gbit/s 不可混用，应明确区分 bit 与 byte。真实交接还包含固定延迟、序列化、布局、同步及链路竞争，不能只用大小/带宽定结论。

服务动作先保护可运行性：显存压力时回退同池或限制接纳；handoff 超预算时比较重算；队列超 SLO 时路由、限流或扩容。每次动作保留触发信号、原因和实际结果。


### 关键路径、I/O 与重叠

```text
CPU 准备 ─────┐
              GPU Prefill ─ KV handoff ─ GPU Decode ─ 输出
下一批 CPU 准备 ────────────┘（有依赖约束的可重叠区间）
```

重叠降低的是关键路径中的暴露等待，不是让各项耗时凭空消失。CPU 双线程、GPU 双流、流水线、DBO 类批次拆分都要证明 overlap 与 idle gap 的变化；并行增加争用或同步也可能更慢。

| 时间位置 | 证据字段 | 解读边界 |
|---|---|---|
| 进入 | queue_wait_ms、input_transfer_bytes/ms | 排队不是 GPU 计算 |
| 执行/输出 | idle_gap、overlap_ratio、stream_output_ms | 定义分母并核对时间线 |
| KV 交接 | transfer_bytes、handoff_ms、recompute_ms | 传输/重算是备选路径，不能重复计入 |
| 网络/通信 | network_wait_ms、communication_ms、exposed_comm_ms | 总通信时间与暴露通信时间不同 |
| 启动 | model_load_ms、cold_start | 不混入热请求平均 |
| 资源池 | Prefill/Decode 利用率、P95/P99 | 一池加速可能让另一池等待 |

接纳控制包括排队、限流/拒绝、允许范围内降级、转移/扩容；必须观察队列长度、拒绝率、超时率、质量和迁移代价。


### 多卡与 MoE 的扩展边界

Tensor/ Pipeline/ Expert Parallel 加入分片、同步、负载均衡与通信。MoE 总参数量不等于每 token 实际激活计算量，要记录 Router 选择的专家数、expert load、capacity factor、token dispatch 和通信空转。

PCIe、NVLink、NVSwitch 的成本取决于实际设备/拓扑。简化链路代理不模拟 All-Reduce/All-Gather 的完整算法与通信量。多 GPU 与自动扩缩容属于进一步的系统扩展，本笔记不作完整展开。


CPU 准备与 GPU 执行若重叠，不能把所有阶段时长直接相加；如果某条通信已被计算覆盖，就应区分总通信与暴露通信。

我们已经有许多可能有效的机制。接下来暂时停止增加机制，面对一张真实记录表：指标有好有坏时，到底能不能说优化成功？


<a id="question-14"></a>

## 14｜吞吐提高了，能否直接宣布优化成功？

先回到第一问的服务目标。批处理重视单位时间完成量，交互服务还关心何时开口、输出是否流畅和尾部请求能否按时完成。同一组结果可能适合一个目标，却不适合另一个目标。

下面保留一份历史 smoke 记录，作为判读练习。阅读时先判断实验改变的是负载还是调度策略，再判断哪些指标可支持结论；不要先被吞吐增长吸引。


### 历史案例：吞吐与交互体验的取舍

以下是**历史 smoke 记录**，非本次运行：Qwen2.5-0.5B-Instruct，RTX 5070 Ti Laptop 12 GB，vLLM 0.11.0，BF16，fixed.jsonl，最大生成长度 64，`max_model_len=2048`、显存利用率配置 0.8、`--enforce-eager`。

| 指标 | concurrency=1 | concurrency=4 |
|---|---:|---:|
| 成功/失败 | 5/0 | 5/0 |
| 请求吞吐 req/s | 3.1189 | 4.2972 |
| 输出吞吐 token/s | 182.1461 | 250.9544 |
| TTFT P50 ms | 33.776 | 234.566 |
| TPOT P50 ms/token | 4.859 | 11.528 |
| E2E P50 ms | 337.176 | 956.383 |
| peak memory | 未采集 | 未采集 |

吞吐提高约 37.8%，TTFT P50 约为原来的 6.94 倍；对批处理可能有价值，对交互请求则需看 SLO。只有 5 个请求，不足以稳定推断尾部；未采集显存，不能得出显存改善结论。该历史环境区分 client PyTorch 2.11.0+cu128 与独立 backend PyTorch 2.8.0+cu128；环境摘要不能把两者混成同一个执行环境。


这张表支持“该小规模记录里，并发增加伴随吞吐与延迟同时上升”，不能支持“某调度算法优于另一算法”。它没有在同一负载下控制策略，也没有足够样本刻画稳定 P99。

所以最后缺的不是更多百分比，而是一套把观察、机制和服务目标连接起来的实验。怎样设计，才不会把巧合或代理指标当成收益？


<a id="question-15"></a>

## 15｜怎样做实验，才能知道究竟是哪项改动有效？

从一个明确假设开始，例如“重复 Prompt 的 TTFT 高，是因为前缀没有复用”。固定模型、请求与到达方式后，比较 cache off/on，同时读取 backend 的命中与复用 token。若 TTFT 降低却没有命中证据，还应检查预热或负载变化；若命中明确但 TTFT 不降，则看维护成本和剩余工作。

同样的结构适用于其他机制：先声明要改变哪个环节，再设置对照，记录直接机制证据，最后检查质量、延迟、吞吐和资源代价。CPU 模拟负责帮助检查公式与状态，GPU 探针负责局部路径，真实服务实验负责用户可见结果；它们接力提供证据，不能相互冒充。

实验组名称只在具体实验内有意义。尤其要把“给系统更多请求”的负载扫描，与“在相同负载下改变策略”的优化对照分开。


### 六步测量协议

1. **固定实验契约**：模型/revision、tokenizer、adapter、backend/version、硬件/驱动/runtime、实际 dtype；请求文件/顺序/到达模式、实际 token 长度、生成与停止设置、cache policy。
2. **预检与基线**：确认配置、依赖、端口和资源；区分 Notebook client 与独立 backend 环境。预热、冷/热状态、测量窗口和重复次数预先确定。
3. **改变一个主要变量**：不同 backend 的比较须承认改变了整个服务栈；避免同时更换模型、长度、dtype 后归因单一机制。
4. **采集原始证据**：请求级时间戳、阶段/队列、成功与失败，机制日志、资源统计；GPU 时间须正确同步，服务端显存不能用仅客户端进程统计代替。
5. **保存独立 JSON 与 manifest**：记录分组、artifact、quality、failure、evidence_level、decision、retest_path；缺失指标用 null/未采集，不能填 0 或代理值。
6. **按目标决策**：质量或可运行性失败则 reject/回退；证据缺失或收益有代价则 tune/补测；同口径、质量与资源/尾延迟均达标才 accept，并保留适用范围。

负的 `candidate−baseline` 延迟差表示改善，正的吞吐差表示改善；吞吐增益比例是 `candidate/baseline−1`，不是原始差值。失败样本单独报告，不与成功运行混合平均，也不能删除后只呈现收益。


### 四类实验分别回答什么

| 项目 | CPU 机制 | G0 | G1 | G2 / 扩展 | 关键门槛 |
|---|---|---|---|---|---|
| 综合基线 | 请求波次、瓶颈分类、决策 | 固定浮点 backend 基线 | 只改并发/batch/dtype/cache/backend 之一 | 已验证组合、SGLang 对照 | 目标服务指标、质量与失败 |
| 投机 | 连续接受、候选排序、质量优先 | target-only | draft+target | proposal length/draft family/budget 中一项 | acceptance、推进、target calls、draft/verify cost |
| 前缀缓存 | 完整块连续命中、LRU、候选选择 | cache off | 同条件 prefix cache | 一项缓存配置或 backend | 直接命中证据、TTFT、维护成本 |
| 调度 | ready batch、状态推进、多指标决策 | concurrency=1 | 增加负载 | 同负载下受控路由/PD 策略 artifact | queue、P99、公平性、池/交接证据 |

各类实验的 G0/G1/G2 命名是局部实验组，含义并不统一。特别是调度实验的 G0→G1 是负载扫描，不能据此宣称“调度策略优化成功”；需要同负载下真实受控的 G2。


### CPU 模型与真实实验分开

综合基线模拟按 `[start,end)` 波次覆盖全部请求，保留最后不足 concurrency 的尾波；显存压力优先分类，其次 Prefill/Decode 占比，再 balanced。同步波次模型不是生产 continuous batching。

调度模拟从已到达 pending 中选不超过 batch_size 的 ready 请求，FIFO/shortest/priority 都保留稳定 tie-breaker。active batch 的教学时间是批内最长 service_ms 除以 batch_speedup，不能相加；完成请求只记录一次。queue wait 被用作首 token 前排队代理，不能当作完整真实 TTFT；fairness/utilization 也是特定模型定义。

| 证据类型 | 可以证明 | 不能直接证明 |
|---|---|---|
| CPU 公式/模拟 | 状态、边界、概率、容量守恒 | GPU 延迟/吞吐 |
| environment_preflight / dry_run | 配置与环境可检查 | 策略实际生效 |
| synthetic_gpu_* | 合成张量或人工负载行为 | 真实模型/服务收益 |
| 真实模型状态上的模拟 | 层/KV 的字节、局部误差、处理时间 | 完整 GPTQ/AWQ/FP8 kernel 或部署 |
| real backend smoke | 小规模服务可运行与初步现象 | 稳定 P99、全面质量和泛化结论 |
| 固定 workload 重复服务实验 | 当前硬件/版本/负载下的服务表现 | 任意模型、硬件或请求分布 |


### 各机制缺一不可的证据

| 候选 | 只有这个数字还不够 | 还需提供 |
|---|---|---|
| FlashAttention | TTFT 下降 | 实际 kernel、语义一致、工作集/时间 |
| 投机解码 | acceptance 高 | 有效输出、draft/verify 成本、目标调用、质量 |
| Prefix Cache | TTFT 下降 | backend hit/reused tokens 与来源、驱逐/占用 |
| 量化 | 文件变小或能加载 | kernel、运行显存、速度、任务质量 |
| PD | throughput 上升 | 同资源同负载、handoff、池平衡、P99 |
| 调度 | GPU 利用率升高 | 等待、完成率、公平性、P99 与策略生效日志 |


**检查理解：**若候选方案质量不达标，吞吐再高也不能按原服务目标接受；若缺失显存数据，就记录未采集，而不是把它填成零。

至此可以把全部推导反过来使用：面对新现象，不必从头尝试所有技术，先选能排除一个关键假设的最小证据。


<a id="question-16"></a>

## 16｜面对新的慢请求，第一项证据该查什么？

把整篇笔记当作逐层排除问题的过程：先确定用户在等什么，再定位实际阶段；然后判断是计算、搬运、容量还是排队；最后才选择改变相应成本的机制。下面的表用于查找下一步，不用于凭一个症状直接给瓶颈定论。

例如“长上下文 + TPOT 上升”至少有三条候选解释：历史 KV 更大、实际 batch 或 kernel 路径改变、服务排队增加。分别检查长度与 Cache 账本、实际执行路径、请求时间线，才能缩小范围。只看到一个相关关系就换 backend，会失去解释变化的依据。


| 观察现象 | 优先假设 | 下一项最小证据 | 阅读位置 |
|---|---|---|---|
| Prompt 越长 TTFT 越高 | Prefill/Attention 成本 | 阶段耗时与 dispatch | 第 2–4 问 |
| TTFT 高但 Prefill 占比低 | 排队/输入/批次准备 | queue 与时间戳 | 第 1、11–13 问 |
| TPOT 高、Decode 占比大 | KV 读取/计算/调度 | Cache 长度、kernel、队列 | 第 5–7、11–12 问 |
| 候选多但吞吐不升 | draft/verify 开销过大 | 每轮提交与成本 | 第 10、15 问 |
| 权重装不下 | 权重驻留 | 完整内存账本、loader | 第 8 问 |
| 上下文/并发增长导致 OOM | KV 容量 | 每 token 字节、页利用率 | 第 5–8 问 |
| 重复前缀无收益 | 未命中或维护成本 | hit/reused/eviction | 第 7、15 问 |
| 突发负载 P99 恶化 | 排队/接纳/公平性 | 到达流、queue、失败 | 第 11–15 问 |
| PD 吞吐升、P99 降不下来 | 交接暴露或池不平衡 | handoff/recompute/池利用率 | 第 13 问 |
| 无单点瓶颈 | 多项串行/重叠成本 | 端到端时间线与单变量对照 | 第 15 问 |

容量受限与带宽受限都常被笼统叫 memory-bound，但行动不同：前者先算能否装下，后者先看要搬多少与实际带宽。

复习时应能独立回答：

1. KV 为什么不是 Prefill 后的独立阶段？TTFT 为什么不等于 Prefill 时间？
2. GQA、FlashAttention、PagedAttention、Prefix Cache、KV 量化分别改变哪个对象？
3. Online Softmax 为什么必须重标定旧的分母和输出？
4. Top-p 边界、Top-K 同分、temperature=0 的接口约定分别有什么风险？
5. 投机拒绝后为何不能继续接受后面的相同 token？bonus 与 correction 有何区别？
6. 块表 OOM 时应保持哪些状态不变？为何 `(位置,当前块)` 不足以唯一标识真实 KV？
7. 权重 MSE 小为什么仍需要任务质量回归？W8A16 教学前向为何可能省存储却不加速？
8. 分池为何可能提高吞吐却伤害 P99？应选择传输、重算还是同池？
9. 调度实验的负载扫描为何不能当作策略对照？只有 TTFT 变化能否证明 Prefix Cache 命中？
10. 历史 smoke、CPU 模拟、真实 backend 重复实验分别允许得出多强的结论？


**完成这条问题链后，应能解释一项优化的完整因果关系：**它改变了哪个对象，减少了什么成本，增加了什么代价，用什么证据证明机制生效，以及在什么 workload 下值得采用。

下面的自检把最容易混淆的四类边界落实成可运行断言：时间口径、KV 字节、跨页增长和前缀身份。


## 阅读与运行约定

公式统一使用 Notebook 数学分隔符；字节计算显式计入 dtype，区分 GB/GiB、Gbit/s/GB/s，以及量化存储与真实低精度执行。教学模拟的能力边界保留在相关问题内。

四张机制图通过同级 `images/` 文件夹引用；上传 GitHub 或下载本地时，请保留该相对位置。下方只运行标准库自检，不启动模型或服务。


<a id="selfcheck"></a>

## 最小可运行自检

仅使用 Python 标准库；下方输出是本次实际自检结果，不是 GPU benchmark。

In [1]:
from math import isfinite

def request_metrics(start_ms, token_ms):
    if not isfinite(start_ms) or not token_ms or any(not isfinite(x) for x in token_ms):
        raise ValueError("需要有限的开始时刻和非空 token 时间戳")
    if token_ms[0] < start_ms or any(b < a for a, b in zip(token_ms, token_ms[1:])):
        raise ValueError("时间戳必须按非递减顺序排列，且不早于请求")
    return {"ttft_ms": token_ms[0] - start_ms,
            "e2e_to_last_token_ms": token_ms[-1] - start_ms,
            "tpot_ms": (token_ms[-1] - token_ms[0]) / (len(token_ms)-1) if len(token_ms)>1 else None}

def kv_bytes(layers, kv_heads, head_dim, lengths, dtype_bytes=2):
    if any(type(x) is not int or x <= 0 for x in (layers, kv_heads, head_dim, dtype_bytes)):
        raise ValueError("模型维度与 dtype_bytes 必须为正整数")
    if any(type(x) is not int or x < 0 for x in lengths):
        raise ValueError("请求长度必须为非负整数")
    return 2 * layers * kv_heads * head_dim * sum(lengths) * dtype_bytes

def block_growth(old_len, new_len, block_size):
    if any(type(x) is not int for x in (old_len, new_len, block_size)) or not (0 <= old_len <= new_len and block_size > 0):
        raise ValueError("需要整数长度、new>=old>=0 和正 block_size")
    return (new_len + block_size-1)//block_size - (old_len+block_size-1)//block_size

def demo():
    metrics = request_metrics(0, [100, 120, 140, 160])
    assert metrics == {"ttft_ms": 100, "e2e_to_last_token_ms": 160, "tpot_ms": 20}
    assert request_metrics(0, [100])["tpot_ms"] is None
    try:
        request_metrics(0, [2, 1])
    except ValueError:
        pass
    else:
        raise AssertionError("未拒绝逆序时间戳")
    assert kv_bytes(32, 8, 128, [4096]) == 512 * 1024**2
    assert kv_bytes(32, 32, 128, [4096]) == 4 * kv_bytes(32, 8, 128, [4096])
    assert block_growth(0, 1, 1) == 1  # block_size=1 的边界
    assert block_growth(15, 17, 16) == 1
    assert block_growth(16, 16, 16) == 0
    # (块位置, 当前块) 会把不同历史混在一起：AB 与 CD 不应允许复用 AD。
    paths = [("A", "B"), ("C", "D")]
    weak_keys = {(i, block) for path in paths for i, block in enumerate(path)}
    query = ("A", "D")
    assert all((i, block) in weak_keys for i, block in enumerate(query))
    full_prefix_keys = {path[:i+1] for path in paths for i in range(len(path))}
    assert query not in full_prefix_keys
    print("PASS：时间口径、单 token 边界、KV 字节、分页边界、前缀上下文反例")

demo()


PASS：时间口径、单 token 边界、KV 字节、分页边界、前缀上下文反例
